# Annotated bottom-table post-processing notebook

This notebook reads OCR-extracted bottom-table results, cleans vote counts from both figure and word fields, compares the two sources, and writes a processed CSV with a final selected vote count.

The annotations explain the purpose of each step without changing the original workflow.


In [ ]:
# Install helper libraries used for vote-word cleanup and fuzzy correction.
# word2number: converts words to numbers, though this notebook mainly uses custom conversion logic.
# pyspellchecker / symspellpy: help correct OCR/spelling errors in vote words.
# num2words: generates valid number words used to build the correction dictionary.
!pip install word2number
!pip install pyspellchecker
!pip install num2words
!pip install symspellpy


  Preparing metadata (setup.py) ... done
  Created wheel for word2number: filename=word2number-1.1-py3-none-any.whl size=5568 sha256=1bcedd48630df27edc657114f8edf9727a5a2a0788b7db644e2c9b31187570c6
  Stored in directory: /root/.cache/pip/wheels/84/ff/26/d3cfbd971e96c5aa3737ecfced81628830d7359b55fbb8ca3b
Successfully built word2number
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 20.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.3/143.3 kB 436.7 kB/s eta 0:00:00
  Created wheel for docopt: filename=docopt-0.6.2-py2.py3-none-any.whl size=13706 sha256=ea9fe7f8645aa73fb0fcf6a271580f2e07a1769726f50c95b1be2100074c4ef4
  Stored in directory: /root/.cache/pip/wheels/fc/ab/d4/5da2067ac95b36618c629a5f93f809425700506f72c9732fac
Successfully built docopt
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.1/144.1 kB 10.1 MB/s eta 0:00:00


In [ ]:
# Import libraries for spell correction, number-word generation, regex cleaning, and dataframe processing.
from spellchecker import SpellChecker
from num2words import num2words
from symspellpy import SymSpell, Verbosity
import re
import pandas as pd


In [ ]:
# Load the extracted bottom-table OCR results.
# on_bad_lines='skip' prevents the whole read from failing if a few CSV rows are malformed.
df = pd.read_csv('bottom_table.csv', on_bad_lines='skip')
# df = pd.read_csv('processed_bottom.csv', on_bad_lines='skip')


In [ ]:
# Standardize column names so later code can use one naming convention.
# This is useful when previous pipeline outputs used slightly different column names.
df.rename(
    columns={
        "Votes in figures": "Vote Score (Figure)",
        'Votes in words': 'Vote Score (Word)',
        'filename': 'image_path'
    },
    inplace=True
)

# Display the dataframe for a quick sanity check.
df


,Political Party,Vote Score (Figure),Vote Score (Word),Signature,confidence,image_path
0,A,NaN,NaN,NaN,[0.91139817237854],../../shared_data/unwrapped_files_jpg/state_17...
1,AA,NaN,NaN,NaN,[0.900286853313446],../../shared_data/unwrapped_files_jpg/state_17...
2,AAC,NaN,NaN,NaN,[0.895061731338501],../../shared_data/unwrapped_files_jpg/state_17...
3,ADC,NaN,NaN,NaN,[0.917688488960266],../../shared_data/unwrapped_files_jpg/state_17...
4,ADP,NaN,one,NaN,"[0.8764908313751221, 0.9147109985351562, 0.807...",../../shared_data/unwrapped_files_jpg/state_17...
...,...,...,...,...,...,...
164074,NRM,0,zero,NaN,"[0.891290545463562, 0.870449423789978, 0.87082...",../../shared_data/unwrapped_files_jpg/state_26...
164075,PDP,0,zero,True,"[0.8976375460624695, 0.8961149454116821, 0.833...",../../shared_data/unwrapped_files_jpg/state_26...
164076,SDP,0,zero,NaN,"[0.8869267106056213, 0.8889849781990051, 0.873...",../../shared_data/unwrapped_files_jpg/state_26...
164077,YPP,0,zero,NaN,"[0.8883392810821533, 0.8867216110229492, 0.871...",../../shared_data/unwrapped_files_jpg/state_26...


In [ ]:
# Build a vocabulary of valid English number words from 0 to 1000.
# This vocabulary is used to correct OCR mistakes in the vote words column.
def generate_cardinal_number_words(start, end):
    number_words_set = set()
    for number in range(start, end + 1):
        # Convert each number into words, e.g. 125 -> "one hundred and twenty-five".
        words = num2words(number)

        # Normalize generated words to match the style expected from OCR text.
        # Example: "twenty-five" -> "twenty five"; remove "and".
        words = words.replace('-', ' ').replace(' and', '')
        number_words_set.update(words.split())
    return number_words_set

start_number = 0
end_number = 1000  # Adjust upward if a polling unit can have vote counts above 1000.

number_words_set = generate_cardinal_number_words(start_number, end_number)

# Add common OCR / handwritten variants for zero votes.
custom_words = {'nil', 'nill'}
number_words_set.update(custom_words)

# Initialize a spellchecker with only valid number-related words.
# This makes corrections more targeted than using a general English dictionary.
spell = SpellChecker()
spell.word_frequency.load_words(number_words_set)

# Initialize SymSpell for fuzzy matching and word segmentation.
# max_dictionary_edit_distance=3 allows fairly noisy OCR words to still match a known number word.
sym_spell = SymSpell(max_dictionary_edit_distance=3, prefix_length=7)
for word in number_words_set:
    sym_spell.create_dictionary_entry(word, 1)


def normalize_word(word):
    # Lowercase and remove non-letter characters before matching.
    # Example: "Tw0!!" would be reduced to letters only, though numeric-looking OCR errors are handled elsewhere.
    word = word.lower()
    word = re.sub(r'[^a-z]', '', word)
    return word


In [ ]:
def match_individual_words(input_word):
    # Correct each whitespace-separated word independently.
    # Best for inputs that already have spaces, e.g. "thre hundrd" -> "three hundred".
    try:
        words = input_word.lower().split()
        corrected_words = []
        for word in words:
            normalized = normalize_word(word)

            # Find the closest known number word within edit distance 2.
            suggestions = sym_spell.lookup(normalized, Verbosity.CLOSEST, max_edit_distance=2)
            if suggestions:
                corrected_words.append(suggestions[0].term)
            elif normalized in number_words_set:
                corrected_words.append(normalized)

        result = ' '.join(corrected_words)
        return result
    except Exception as e:
        print(f"Error processing word '{input_word}' in match_individual_words: {str(e)}")
        return input_word


def use_word_segmentation(input_word):
    # Correct a full string by trying to split/segment it into known words.
    # Best for OCR text where words may be joined together, e.g. "onehundredtwo".
    try:
        normalized = normalize_word(input_word)
        suggestions = sym_spell.word_segmentation(normalized)
        result = suggestions.corrected_string
        return result
    except Exception as e:
        print(f"Error processing word '{input_word}' in use_word_segmentation: {str(e)}")
        return input_word


In [ ]:
def clean_vote_string_match(vote_string):
    # Clean the vote-word field using individual-word fuzzy matching.
    # Missing or empty vote words are treated as zero.
    if pd.isna(vote_string):
        return "zero"

    vote_string = str(vote_string).strip()

    if not vote_string:
        return "zero"

    result = match_individual_words(vote_string)

    # Treat "nil" / "nill" as zero votes.
    if 'nil' in result:
        result = "zero"
    return result


def clean_vote_string_segment(vote_string):
    # Clean the vote-word field using SymSpell word segmentation.
    # This is a second strategy that may work better when OCR merges words.
    if pd.isna(vote_string):
        return "zero"

    vote_string = str(vote_string).strip()

    if not vote_string:
        return "zero"

    result = use_word_segmentation(vote_string)

    # Treat "nil" / "nill" as zero votes.
    if 'nil' in result:
        result = "zero"
    return result


# Apply both cleaning strategies to the vote words.
# Keeping both columns lets us later compare which strategy agrees with the numeric vote field.
df['cleaned_votes_match'] = df['Vote Score (Word)'].apply(clean_vote_string_match)
df['cleaned_votes_segment'] = df['Vote Score (Word)'].apply(clean_vote_string_segment)


Error processing word '12010' in use_word_segmentation: list index out of range
Error processing word '2020' in use_word_segmentation: list index out of range
Error processing word '242' in use_word_segmentation: list index out of range
Error processing word '11' in use_word_segmentation: list index out of range
Error processing word '2020' in use_word_segmentation: list index out of range
Error processing word '2010' in use_word_segmentation: list index out of range
Error processing word '2000' in use_word_segmentation: list index out of range
Error processing word '2010' in use_word_segmentation: list index out of range
Error processing word '2000' in use_word_segmentation: list index out of range
Error processing word '2023' in use_word_segmentation: list index out of range
Error processing word '1)' in use_word_segmentation: list index out of range
Error processing word '11' in use_word_segmentation: list index out of range
Error processing word '11' in use_word_segmentation: list 

In [ ]:
def safe_process(func):
    # Wrap a function so one bad row does not stop the whole dataframe processing step.
    # Returns None when an individual value cannot be processed.
    def wrapper(x):
        try:
            return func(x)
        except Exception as e:
            print(f"Error processing {x}: {str(e)}")
            return None
    return wrapper


In [ ]:
import pandas as pd
import re
import ast

# This section converts cleaned vote words into numeric values.
# It uses custom logic because OCR output can be irregular, such as:
#   - normal words: 'one hundred twenty three'
#   - digit-like words: 'one two three'
#   - strings that contain an actual number: '123'
#   - list-like strings from earlier extraction steps: ['one hundred']

# Define number words
units = [
    'zero', 'one', 'two', 'three', 'four', 'five', 'six', 'seven', 'eight',
    'nine'
]
teens = [
    'ten', 'eleven', 'twelve', 'thirteen', 'fourteen', 'fifteen', 'sixteen',
    'seventeen', 'eighteen', 'nineteen'
]
tens = [
    'twenty', 'thirty', 'forty', 'fifty', 'sixty', 'seventy', 'eighty',
    'ninety'
]
hundred = ['hundred']

# Create a mapping from number words to their numeric values
numwords = {}
for idx, word in enumerate(units):
    numwords[word] = idx
for idx, word in enumerate(teens):
    numwords[word] = 10 + idx
for idx, word in enumerate(tens):
    numwords[word] = 10 * (idx + 2)
numwords['hundred'] = 100

def clean_string(s):
    # Normalize text before number conversion.
    # Remove special characters and extra spaces
    s = re.sub(r'[^a-zA-Z0-9\s]', '', s)
    # Replace multiple spaces with a single space
    s = re.sub(r'\s+', ' ', s)
    return s.strip().lower()

def word_to_num(number_sentence):
    # Convert one cleaned number phrase into an integer where possible.
    if not isinstance(number_sentence, str):
        return ''  # Return empty string for non-string inputs
    # if number_sentence.strip()=='':
    #     return ''
    stripped_sentence = number_sentence.strip()
    # Check if the stripped sentence is empty
    if stripped_sentence == '':
        return ''
    if stripped_sentence == '-':
        return 0

    # Check for existing numeric value
    numeric_match = re.search(r'\d+', number_sentence)
    if numeric_match:
        return int(numeric_match.group())

    number_sentence = number_sentence.lower()
    number_sentence = number_sentence.replace('-', ' ')
    words = number_sentence.strip().split()

    # Check if it's a simple sequence of individual numbers like 'one zero six'
    if all(word in units for word in words):
        return int(''.join([str(numwords[word]) for word in words]))

    total_number = 0
    i = 0

    if len(words) >= 1:
        first_word = words[0]
        if first_word in units:
            first_value = numwords[first_word]
            if len(words) >= 2:
                second_word = words[1]
                if second_word == 'hundred':
                    # Handle cases like 'one hundred twenty'
                    current = first_value * 100
                    i = 2
                    rest_number_str = ''
                    while i < len(words):
                        word = words[i]
                        if word in numwords:
                            # Handle tens and units
                            if word in tens:
                                value = numwords[word]
                                if i + 1 < len(words) and words[i + 1] in units:
                                    value += numwords[words[i + 1]]
                                    i += 1
                                rest_number_str += str(value)
                            else:
                                rest_number_str += str(numwords[word])
                            i += 1
                        else:
                            return ''  # Return empty string for invalid words
                    if rest_number_str == '':
                        rest_number = 0
                    else:
                        rest_number = int(rest_number_str)
                    total_number = current + rest_number
                else:
                    # Handle cases like 'seven eight' (78)
                    if second_word in units:
                        total_number = first_value * 10 + numwords[second_word]
                        i = 2
                    else:
                        # Treat first_value as hundreds digit
                        current = first_value * 100
                        rest_number_str = ''
                        i = 1
                        while i < len(words):
                            word = words[i]
                            if word in numwords:
                                # Handle tens and units
                                if word in tens:
                                    value = numwords[word]
                                    if i + 1 < len(words) and words[i + 1] in units:
                                        value += numwords[words[i + 1]]
                                        i += 1
                                    rest_number_str += str(value)
                                else:
                                    rest_number_str += str(numwords[word])
                                i += 1
                            else:
                                return ''  # Return empty string for invalid words
                        if rest_number_str == '':
                            rest_number = 0
                        else:
                            rest_number = int(rest_number_str)
                        total_number = current + rest_number
            else:
                # Single unit word
                total_number = first_value
        elif first_word == 'hundred':
            # Handle case where 'hundred' is the first word
            current = 100
            i = 1
            rest_number_str = ''
            while i < len(words):
                word = words[i]
                if word in numwords:
                    rest_number_str += str(numwords[word])
                    i += 1
                else:
                    return ''  # Return empty string for invalid words
            if rest_number_str == '':
                rest_number = 0
            else:
                rest_number = int(rest_number_str)
            total_number = current + rest_number
        else:
            # First word is not a unit, process as number less than 100
            rest_number_str = ''
            while i < len(words):
                word = words[i]
                if word in numwords:
                    # Handle tens and units
                    if word in tens:
                        value = numwords[word]
                        if i + 1 < len(words) and words[i + 1] in units:
                            value += numwords[words[i + 1]]
                            i += 1
                        rest_number_str += str(value)
                    else:
                        rest_number_str += str(numwords[word])
                    i += 1
                else:
                    return ''  # Return empty string for invalid words
            total_number = int(rest_number_str) if rest_number_str else -1
    else:
        return ''  # Return empty string for empty input

    return total_number if total_number != -1 else ''

def process_vote_array(vote_array):
    # Convert a list-like vote value into a list of numeric vote candidates.
    if isinstance(vote_array, str):
        try:
            # Convert string representation of list to actual list
            vote_array = ast.literal_eval(vote_array)
        except:
            # If conversion fails, treat the input as a single item
            vote_array = [vote_array]

    # Process each item in the array
    numeric_votes = [word_to_num(str(item)) for item in vote_array]

    return numeric_votes


def process_single_vote(vote_item):
    # Convert one vote value into a single numeric value.
    # If the input is a list-like string, it uses the first list item.
    if isinstance(vote_item, str):
        try:
            # Check if the string is actually a representation of a list
            vote_list = ast.literal_eval(vote_item)
            if isinstance(vote_list, list):
                # If it's a list, process the first item
                return word_to_num(str(vote_list[0])) if vote_list else None
            else:
                # If it's not a list, process it as a single item
                return word_to_num(vote_item)
        except:
            # If conversion fails, treat the input as a single item
            return word_to_num(vote_item)
    else:
        # If it's not a string, convert to string and process
        return word_to_num(str(vote_item))

# Load your DataFrame (assuming it's already loaded as 'df')
# df['num_votes_match'] = df['cleaned_votes_match'].apply(process_single_vote)
# df['num_votes_segment'] = df['cleaned_votes_segment'].apply(process_single_vote)

In [ ]:
# Convert the cleaned vote-word strings into numeric vote counts.
# The safe wrapper means errors become None instead of stopping the notebook.
df['num_votes_match'] = df['cleaned_votes_match'].apply(safe_process(process_single_vote))
df['num_votes_segment'] = df['cleaned_votes_segment'].apply(safe_process(process_single_vote))


In [ ]:
import re
import pandas as pd


def clean_number(x):
    # Extract numeric candidates from the OCR "Vote Score (Figure)" field.
    # Returns up to three possible values:
    #   cleaned_number_1: first number found
    #   cleaned_number_2: second number found, if any
    #   cleaned_number_3: concatenation of first + second, useful when OCR splits digits
    # Example: "1 23" can produce 1, 23, and 123.
    if pd.isna(x):
        return pd.NA, pd.NA, pd.NA

    x = str(x)

    # Replace common OCR confusions with likely digits.
    replacements = {
        'I': '1', 'l': '1', '/': '1',
        '\\': '1',
        '&': '8',
        's': '5',
        'o': '0', 'O': '0',
        'Z': '2'
    }

    for old, new in replacements.items():
        x = x.replace(old, new)

    # Extract standalone numbers, avoiding digits embedded inside words.
    numbers = re.findall(r"(?<!\w)(\d+(?:\.\d+)?)(?!\w)", x)

    # Remove decimal points because OCR may read separated digits as decimals.
    numbers = [num.replace('.', '') for num in numbers]

    # Convert extracted strings to numeric values.
    numbers = [pd.to_numeric(num, errors='coerce') for num in numbers]

    cleaned_number_1 = pd.NA
    cleaned_number_2 = pd.NA
    cleaned_number_3 = pd.NA

    if len(numbers) == 0:
        return pd.NA, pd.NA, pd.NA
    elif len(numbers) == 1:
        cleaned_number_1 = numbers[0]
    else:
        cleaned_number_1 = numbers[0]
        cleaned_number_2 = numbers[1]
        cleaned_number_3 = int(f"{int(cleaned_number_1)}{int(cleaned_number_2)}")

    return cleaned_number_1, cleaned_number_2, cleaned_number_3


In [ ]:
# Keep only the four major parties needed for the final bottom-table summary.
df_filter = df[df['Political Party'].isin(['APC', 'PDP', 'LP', 'NNPP'])]
df_filter.reset_index(inplace=True, drop=True)

# First try to extract numeric candidates from the vote-figure field.
df_filter[['cleaned_number_1', 'cleaned_number_2', 'cleaned_number_3']] = (
    df_filter["Vote Score (Figure)"].apply(clean_number).tolist()
)

# If the figure field is missing, treat the first numeric candidate as zero.
df_filter.loc[df_filter['Vote Score (Figure)'].isna(), 'cleaned_number_1'] = 0

# If no number could be extracted from the figure field, try extracting from the vote-word field.
df_filter.loc[
    df_filter['cleaned_number_1'].isna(),
    ['cleaned_number_1', 'cleaned_number_2', 'cleaned_number_3']
] = df_filter.loc[
    df_filter['cleaned_number_1'].isna(),
    'Vote Score (Word)'
].apply(clean_number).tolist()

# If still no number was found, check whether the figure field represents "nil"/zero after word cleanup.
df_filter.loc[df_filter['cleaned_number_1'].isna(), 'temp'] = (
    df_filter.loc[df_filter['cleaned_number_1'].isna(), 'Vote Score (Figure)'].apply(clean_vote_string_match)
)
df_filter.loc[df_filter['temp'] == 'zero', 'cleaned_number_1'] = 0


<ipython-input-14-397757492104>:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filter[['cleaned_number_1', 'cleaned_number_2', 'cleaned_number_3']] = df_filter["Vote Score (Figure)"].apply(clean_number).tolist()
<ipython-input-14-397757492104>:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filter[['cleaned_number_1', 'cleaned_number_2', 'cleaned_number_3']] = df_filter["Vote Score (Figure)"].apply(clean_number).tolist()
<ipython-input-14-397757492104>:3: SettingWithCopyWarning: 
A value is trying 

In [ ]:
# Convert the vote-word-derived counts to numeric values so they can be compared with figure-derived candidates.
df_filter['num_votes_match_numeric'] = pd.to_numeric(df_filter['num_votes_match'], errors='coerce')
df_filter['num_votes_segment_numeric'] = pd.to_numeric(df_filter['num_votes_segment'], errors='coerce')

# Optional debugging code for reviewing rows where both word-cleaning strategies failed.
# non_numeric_mask = (df_filter['num_votes_match_numeric'].isna()) & (df_filter['num_votes_segment_numeric'].isna())
# non_numeric_rows = df_filter[non_numeric_mask]

# Fill missing figure-derived candidates with sentinel value -50.
# Sentinel values make it easier to compare rows without mixing true missing values with valid vote counts.
df_filter.loc[:, 'cleaned_number_1'] = df_filter['cleaned_number_1'].fillna(-50)
df_filter.loc[:, 'cleaned_number_2'] = df_filter['cleaned_number_2'].fillna(-50)

# NOTE: This line existed in the original notebook. It fills cleaned_number_3 using cleaned_number_2.
# If this was accidental, change it to: df_filter['cleaned_number_3'].fillna(-50)
df_filter.loc[:, 'cleaned_number_3'] = df_filter['cleaned_number_2'].fillna(-50)

# Fill failed word-derived numeric values with sentinel value -15, then convert to integer.
df_filter.loc[:, 'num_votes_match_numeric'] = df_filter['num_votes_match_numeric'].fillna(-15)
df_filter.loc[:, 'num_votes_match_numeric'] = df_filter['num_votes_match_numeric'].astype('int64')

df_filter.loc[:, 'num_votes_segment_numeric'] = df_filter['num_votes_segment_numeric'].fillna(-15)
df_filter.loc[:, 'num_votes_segment_numeric'] = df_filter['num_votes_segment_numeric'].astype('int64')


<ipython-input-15-f34c9dec0cfc>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filter['num_votes_match_numeric'] = pd.to_numeric(df_filter['num_votes_match'], errors='coerce')
<ipython-input-15-f34c9dec0cfc>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filter['num_votes_segment_numeric'] = pd.to_numeric(df_filter['num_votes_segment'], errors='coerce')
<ipython-input-15-f34c9dec0cfc>:8: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a fu

In [ ]:
import pandas as pd
import numpy as np


def numbers_match(a, b):
    """
    Check whether two vote counts are close enough to be treated as matching.

    Matching rules:
    1. Exact match, e.g. 123 == 123.
    2. Difference is a multiple of 100, which can handle OCR errors that add/drop a hundreds digit.
    3. Difference is 2 or less, which allows small OCR/recognition mistakes.
    """
    try:
        a = int(a)
        b = int(b)
        if a == b:
            return True
        elif abs(a - b) % 100 == 0:
            return True
        elif abs(a - b) <= 2:
            return True
        else:
            return False
    except:
        return False


def find_real_number(row):
    """
    Choose the final vote count for one row by comparing OCR-derived candidates.

    The logic is:
    1. Collect valid numeric candidates from the vote-figure field.
    2. Compare them against the number converted from cleaned vote words.
    3. Prefer the individual-word correction result first.
    4. If that does not match, try the segmentation correction result.
    5. Return NaN if no candidate agrees.
    """
    def is_valid_num(x):
        # -15 and -50 are sentinel values introduced earlier for failed/missing values.
        return pd.notnull(x) and x != -15 and x != -50

    cleaned_numbers = []
    if is_valid_num(row['cleaned_number_1']):
        cleaned_numbers.append(int(row['cleaned_number_1']))
    if is_valid_num(row['cleaned_number_2']):
        cleaned_numbers.append(int(row['cleaned_number_2']))
    if is_valid_num(row['cleaned_number_3']):
        cleaned_numbers.append(int(row['cleaned_number_3']))

    # First compare figure-derived candidates with the word count from individual-word matching.
    if is_valid_num(row['num_votes_match_numeric']):
        num_votes_match = int(row['num_votes_match_numeric'])
        for cleaned_num in cleaned_numbers:
            if numbers_match(num_votes_match, cleaned_num):
                return cleaned_num

    # Then compare figure-derived candidates with the word count from word segmentation.
    if is_valid_num(row['num_votes_segment_numeric']):
        num_votes_segment = int(row['num_votes_segment_numeric'])
        for cleaned_num in cleaned_numbers:
            if numbers_match(num_votes_segment, cleaned_num):
                return cleaned_num

    return np.nan


In [ ]:
# Apply the matching logic row by row to produce the final selected vote count.
df_filter['real'] = df_filter.apply(find_real_number, axis=1)

# Extract a shorter file identifier from the image path.
# [:-11] appears to remove a suffix such as "_crop.jpg" from the filename.
df_filter['file_name'] = df_filter['image_path'].apply(lambda x: x.split('/')[-1][:-11])


<ipython-input-17-ddb0d011bdd1>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filter['real'] = df_filter.apply(find_real_number, axis=1)
<ipython-input-17-ddb0d011bdd1>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filter['file_name']=df_filter['image_path'].apply(lambda x:x.split('/')[-1][:-11])


In [ ]:
# Display the processed dataframe for inspection.
df_filter


,Political Party,Vote Score (Figure),Vote Score (Word),Signature,confidence,image_path,cleaned_votes_match,cleaned_votes_segment,num_votes_match,num_votes_segment,cleaned_number_1,cleaned_number_2,cleaned_number_3,temp,num_votes_match_numeric,num_votes_segment_numeric,real,file_name
0,APC,88,Eigty eigt,True,"[0.8458033204078674, 0.6791602969169617, 0.552...",../../shared_data/unwrapped_files_jpg/state_17...,eighty eight,eighty eight,88,88,88,-50,-50,NaN,88.0,88.0,88.0,17_21_05_012
1,LP,NaN,NaN,NaN,[0.9213324189186096],../../shared_data/unwrapped_files_jpg/state_17...,zero,zero,0,0,0,-50,-50,NaN,0.0,0.0,0.0,17_21_05_012
2,NNPP,7,Seven,NaN,"[0.8914812803268433, 0.903925359249115, 0.7960...",../../shared_data/unwrapped_files_jpg/state_17...,seven,seven,7,7,7,-50,-50,NaN,7.0,7.0,7.0,17_21_05_012
3,PDP,23,Twenty three,NaN,"[0.8464422225952148, 0.7800012826919556, 0.918...",../../shared_data/unwrapped_files_jpg/state_17...,twenty three,twenty three,23,23,23,-50,-50,NaN,23.0,23.0,23.0,17_21_05_012
4,APC,48,forts Eight,True,"[0.8236913681030273, 0.8279861807823181, 0.879...",../../shared_data/unwrapped_files_jpg/state_24...,forty eight,forty eight,48,48,48,-50,-50,NaN,48.0,48.0,48.0,24_19_08_009
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38650,PDP,4,FOUR,True,"[0.8402000665664673, 0.7280642986297607, 0.880...",../../shared_data/unwrapped_files_jpg/state_28...,four,four,4,4,4,-50,-50,NaN,4.0,4.0,4.0,28_16_07_066
38651,APC,0,zero,NaN,"[0.9043877720832824, 0.8730465173721313, 0.849...",../../shared_data/unwrapped_files_jpg/state_26...,zero,zero,0,0,0,-50,-50,NaN,0.0,0.0,0.0,26_03_10_003
38652,LP,9,zero,NaN,"[0.8987525701522827, 0.8909909129142761, 0.833...",../../shared_data/unwrapped_files_jpg/state_26...,zero,zero,0,0,9,-50,-50,NaN,0.0,0.0,NaN,26_03_10_003
38653,NNPP,0,zero,True,"[0.892021656036377, 0.8781155347824097, 0.8706...",../../shared_data/unwrapped_files_jpg/state_26...,zero,zero,0,0,0,-50,-50,NaN,0.0,0.0,0.0,26_03_10_003


In [ ]:
# Save the processed bottom-table results.
df_filter.to_csv('./summary_abhik/processed_bottom.csv', index=False)


In [ ]:
# Display the final dataframe again after saving.
df_filter


,Political Party,Vote Score (Figure),Vote Score (Word),Signature,confidence,image_path,cleaned_votes_match,cleaned_votes_segment,num_votes_match,num_votes_segment,cleaned_number_1,cleaned_number_2,cleaned_number_3,temp,num_votes_match_numeric,num_votes_segment_numeric,real,file_name
0,APC,88,Eigty eigt,True,"[0.8458033204078674, 0.6791602969169617, 0.552...",../../shared_data/unwrapped_files_jpg/state_17...,eighty eight,eighty eight,88,88,88,-50,-50,NaN,88.0,88.0,88.0,17_21_05_012
1,LP,NaN,NaN,NaN,[0.9213324189186096],../../shared_data/unwrapped_files_jpg/state_17...,zero,zero,0,0,0,-50,-50,NaN,0.0,0.0,0.0,17_21_05_012
2,NNPP,7,Seven,NaN,"[0.8914812803268433, 0.903925359249115, 0.7960...",../../shared_data/unwrapped_files_jpg/state_17...,seven,seven,7,7,7,-50,-50,NaN,7.0,7.0,7.0,17_21_05_012
3,PDP,23,Twenty three,NaN,"[0.8464422225952148, 0.7800012826919556, 0.918...",../../shared_data/unwrapped_files_jpg/state_17...,twenty three,twenty three,23,23,23,-50,-50,NaN,23.0,23.0,23.0,17_21_05_012
4,APC,48,forts Eight,True,"[0.8236913681030273, 0.8279861807823181, 0.879...",../../shared_data/unwrapped_files_jpg/state_24...,forty eight,forty eight,48,48,48,-50,-50,NaN,48.0,48.0,48.0,24_19_08_009
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38650,PDP,4,FOUR,True,"[0.8402000665664673, 0.7280642986297607, 0.880...",../../shared_data/unwrapped_files_jpg/state_28...,four,four,4,4,4,-50,-50,NaN,4.0,4.0,4.0,28_16_07_066
38651,APC,0,zero,NaN,"[0.9043877720832824, 0.8730465173721313, 0.849...",../../shared_data/unwrapped_files_jpg/state_26...,zero,zero,0,0,0,-50,-50,NaN,0.0,0.0,0.0,26_03_10_003
38652,LP,9,zero,NaN,"[0.8987525701522827, 0.8909909129142761, 0.833...",../../shared_data/unwrapped_files_jpg/state_26...,zero,zero,0,0,9,-50,-50,NaN,0.0,0.0,NaN,26_03_10_003
38653,NNPP,0,zero,True,"[0.892021656036377, 0.8781155347824097, 0.8706...",../../shared_data/unwrapped_files_jpg/state_26...,zero,zero,0,0,0,-50,-50,NaN,0.0,0.0,0.0,26_03_10_003


In [ ]:
# Empty cell reserved for additional checks or follow-up processing.
